#### Importando Bibliotecas

In [1]:
import requests
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from datetime import datetime

#### Função Auxiliar para Extração dos Dados

In [2]:
def extrair_codigo_fonte_com_rolagem(url):
    try:
        # Inicializa o driver do Chrome
        driver = webdriver.Chrome()
        
        # Abre a URL no navegador
        driver.get(url)
        
        # Aguarda um tempo para a página carregar
        time.sleep(5)
        
        # Inicializa a variável para armazenar o código fonte
        codigo_fonte = ""
        
        # Define o incremento de rolagem
        incremento_rolagem = 800
        
        # Inicializa a posição inicial da rolagem
        posicao_rolagem = 0
        
        while True:
            # Rola a página para baixo em pequenos incrementos
            driver.execute_script("window.scrollTo(0, {});".format(posicao_rolagem))
            
            # Aguarda um curto período de tempo para que os novos elementos sejam carregados
            time.sleep(1)
            
            # Atualiza a posição de rolagem
            posicao_rolagem += incremento_rolagem
            
            # Verifica se a posição de rolagem atingiu o final da página
            if posicao_rolagem >= driver.execute_script("return document.body.scrollHeight"):
                codigo_fonte += driver.page_source
                break
        
        # Fecha o navegador
        driver.quit()
        
        return codigo_fonte
    except Exception as e:
        print("Ocorreu um erro:", e)
        return None

#### 1. Extração dos Dados (Santa Cecília)

Queremos extrair os dados de uma busca fixa, com os seguintes parâmetros:

- Bairro: Santa Cecília
- Cidade: São Paulo - SP
- Tipo: Apartamento residencial, Studio residencial, Kitnet residencial, Flat residencial e Loft residencial 
- Preço Total Mínimo: R\$ 3000,00
- Preço Total Máximo: R\$ 6000,00
- Área Mínima: 45m²
- Área Máxima: 110m²

que geram as seguintes URL: 

**https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+centro+sta-cecilia/?__ab=olx:control,zap-newldp:control,super-high:control,exp-aa-test:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Centro,Santa%20Cec%C3%ADlia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3ECentro%3ESanta%20Cecilia,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=1&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110**

https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+centro+sta-cecilia/?__ab=olx:control,zap-newldp:control,super-high:control,exp-aa-test:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Centro,Santa%20Cec%C3%ADlia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3ECentro%3ESanta%20Cecilia,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=2&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110

In [3]:
# Define o número de páginas contido na busca
paginas = 2

# Inicializar uma lista para armazenar as URLs
urls = []

# Chama a função para extrair o código fonte com rolagem para cada página
for pagina in range(1, paginas+1):
    # URL Busca
    url_busca = f"https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+centro+sta-cecilia/?__ab=olx:control,zap-newldp:control,super-high:control,exp-aa-test:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Centro,Santa%20Cec%C3%ADlia,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3ECentro%3ESanta%20Cecilia,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina={pagina}&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110"

    codigo_fonte = extrair_codigo_fonte_com_rolagem(url_busca)

    print(f"Código fonte da página {pagina} obtido com sucesso!")
    soup = BeautifulSoup(codigo_fonte, 'lxml')

    # Encontrar todos os elementos <div> com o atributo data-position
    divs_com_data_position = soup.find_all('div', {'data-position': True})
    
    # Iterar sobre os elementos <div> encontrados
    for div in divs_com_data_position:
        # Encontrar todos os elementos <a> dentro do <div>
        links = div.find_all('a', href=True)
        # Extrair as URLs e adicioná-las à lista de URLs
        for link in links:
            urls.append(link['href'])
    print(f"URLs da página {pagina} salvas com sucesso!") 

Código fonte da página 1 obtido com sucesso!
URLs da página 1 salvas com sucesso!
Código fonte da página 2 obtido com sucesso!
URLs da página 2 salvas com sucesso!


In [4]:
# Especificando o bairro da busca para salvamento dos dados
bairro = 'santa_cecilia'

# Expondo o tamanho do arquivo de hoje
print(f"Hoje tivemos {len(urls)} URLs!")

# Salvando as URLs retornadas em um dataframe
data = pd.DataFrame({'URL': urls})

# Obtenha a data de hoje
today = datetime.now().strftime('%Y_%m_%d')

# Criar o caminho completo do arquivo
caminho_arquivo = f"{today}/zapimoveis_{bairro}_{today}.csv"

# Criar a pasta se ela não existir
if not os.path.exists(today):
    os.makedirs(today)

# Salvando o dataframe em um arquivo .csv com o formato 'zapimoveis_bairro_YYYY_MM_DD.csv'
data.to_csv(caminho_arquivo, index=False)

Hoje tivemos 142 URLs!


____________________________

#### 2. Extração dos Dados (Perdizes)

Queremos extrair os dados de uma busca fixa, com os seguintes parâmetros:

- Bairro: Perdizes
- Cidade: São Paulo - SP
- Tipo: Apartamento residencial, Studio residencial, Kitnet residencial, Flat residencial e Loft residencial 
- Preço Total Mínimo: R\$ 3000,00
- Preço Total Máximo: R\$ 6000,00
- Área Mínima: 45m²
- Área Máxima: 110m²

que geram as seguintes URLs:

**https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+perdizes/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Perdizes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EPerdizes,-23.536902,-46.674317,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=1&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110**

https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+perdizes/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Perdizes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EPerdizes,-23.536902,-46.674317,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=2&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110

https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+perdizes/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Perdizes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EPerdizes,-23.536902,-46.674317,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=3&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110

https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+perdizes/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Perdizes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EPerdizes,-23.536902,-46.674317,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=4&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110

In [5]:
# Define o número de páginas contido na busca
paginas = 4

# Inicializar uma lista para armazenar as URLs
urls = []

# Chama a função para extrair o código fonte com rolagem para cada página
for pagina in range(1, paginas+1):
    # URL Busca
    url_busca = f"https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+perdizes/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Perdizes,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EPerdizes,-23.536902,-46.674317,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina={pagina}&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110"

    codigo_fonte = extrair_codigo_fonte_com_rolagem(url_busca)

    print(f"Código fonte da página {pagina} obtido com sucesso!")
    soup = BeautifulSoup(codigo_fonte, 'lxml')

    # Encontrar todos os elementos <div> com o atributo data-position
    divs_com_data_position = soup.find_all('div', {'data-position': True})
    
    # Iterar sobre os elementos <div> encontrados
    for div in divs_com_data_position:
        # Encontrar todos os elementos <a> dentro do <div>
        links = div.find_all('a', href=True)
        # Extrair as URLs e adicioná-las à lista de URLs
        for link in links:
            urls.append(link['href'])
    print(f"URLs da página {pagina} salvas com sucesso!") 

Código fonte da página 1 obtido com sucesso!
URLs da página 1 salvas com sucesso!
Código fonte da página 2 obtido com sucesso!
URLs da página 2 salvas com sucesso!
Código fonte da página 3 obtido com sucesso!
URLs da página 3 salvas com sucesso!
Código fonte da página 4 obtido com sucesso!
URLs da página 4 salvas com sucesso!


In [6]:
# Especificando o bairro da busca para salvamento dos dados
bairro = 'perdizes'

# Expondo o tamanho do arquivo de hoje
print(f"Hoje tivemos {len(urls)} URLs!")

# Salvando as URLs retornadas em um dataframe
data = pd.DataFrame({'URL': urls})

# Obtenha a data de hoje
today = datetime.now().strftime('%Y_%m_%d')

# Criar o caminho completo do arquivo
caminho_arquivo = f"{today}/zapimoveis_{bairro}_{today}.csv"

# Criar a pasta se ela não existir
if not os.path.exists(today):
    os.makedirs(today)

# Criar o caminho completo do arquivo
caminho_arquivo = f"{today}/zapimoveis_{bairro}_{today}.csv"

# Criar a pasta se ela não existir
if not os.path.exists(today):
    os.makedirs(today)

# Salvando o dataframe em um arquivo .csv com o formato 'zapimoveis_bairro_YYYY_MM_DD.csv'
data.to_csv(caminho_arquivo, index=False)

Hoje tivemos 279 URLs!


#### 3. Extração dos Dados (Barra Funda)

Queremos extrair os dados de uma busca fixa, com os seguintes parâmetros:

- Bairro: Barra Funda
- Cidade: São Paulo - SP
- Tipo: Apartamento residencial, Studio residencial, Kitnet residencial, Flat residencial e Loft residencial 
- Preço Total Mínimo: R\$ 3000,00
- Preço Total Máximo: R\$ 6000,00
- Área Mínima: 45m²
- Área Máxima: 110m²

que geram as seguintes URLs:

**https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+barra-funda/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Barra%20Funda,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EBarra%20Funda,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=1&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110**

https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+barra-funda/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Barra%20Funda,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EBarra%20Funda,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina=2&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110

In [7]:
# Define o número de páginas contido na busca
paginas = 2

# Inicializar uma lista para armazenar as URLs
urls = []

# Chama a função para extrair o código fonte com rolagem para cada página
for pagina in range(1, paginas+1):
    # URL Busca
    url_busca = f"https://www.zapimoveis.com.br/aluguel/apartamentos/sp+sao-paulo+zona-oeste+barra-funda/?__ab=olx:control,super-high:control,exp-aa-test:control,zap-newldp:control,novopos:new,rp-imob:enabled&transacao=aluguel&onde=,S%C3%A3o%20Paulo,S%C3%A3o%20Paulo,Zona%20Oeste,Barra%20Funda,,,neighborhood,BR%3ESao%20Paulo%3ENULL%3ESao%20Paulo%3EZona%20Oeste%3EBarra%20Funda,-23.529192,-46.660999,&tipos=apartamento_residencial,studio_residencial,kitnet_residencial,flat_residencial,loft_residencial&pagina={pagina}&precoTotalMinimo=3000&precoTotalMaximo=6000&areaMinima=45&areaMaxima=110"

    codigo_fonte = extrair_codigo_fonte_com_rolagem(url_busca)

    print(f"Código fonte da página {pagina} obtido com sucesso!")
    soup = BeautifulSoup(codigo_fonte, 'lxml')

    # Encontrar todos os elementos <div> com o atributo data-position
    divs_com_data_position = soup.find_all('div', {'data-position': True})
    
    # Iterar sobre os elementos <div> encontrados
    for div in divs_com_data_position:
        # Encontrar todos os elementos <a> dentro do <div>
        links = div.find_all('a', href=True)
        # Extrair as URLs e adicioná-las à lista de URLs
        for link in links:
            urls.append(link['href'])
    print(f"URLs da página {pagina} salvas com sucesso!") 

Código fonte da página 1 obtido com sucesso!
URLs da página 1 salvas com sucesso!
Código fonte da página 2 obtido com sucesso!
URLs da página 2 salvas com sucesso!


In [8]:
# Especificando o bairro da busca para salvamento dos dados
bairro = 'barra_funda'

# Expondo o tamanho do arquivo de hoje
print(f"Hoje tivemos {len(urls)} URLs!")

# Salvando as URLs retornadas em um dataframe
data = pd.DataFrame({'URL': urls})

# Obtenha a data de hoje
today = datetime.now().strftime('%Y_%m_%d')

# Criar o caminho completo do arquivo
caminho_arquivo = f"{today}/zapimoveis_{bairro}_{today}.csv"

# Criar a pasta se ela não existir
if not os.path.exists(today):
    os.makedirs(today)

# Criar o caminho completo do arquivo
caminho_arquivo = f"{today}/zapimoveis_{bairro}_{today}.csv"

# Criar a pasta se ela não existir
if not os.path.exists(today):
    os.makedirs(today)

# Salvando o dataframe em um arquivo .csv com o formato 'zapimoveis_bairro_YYYY_MM_DD.csv'
data.to_csv(caminho_arquivo, index=False)

Hoje tivemos 140 URLs!
